In [5]:
from google.colab import files
uploaded = files.upload()

Saving archive (1).zip to archive (1).zip


In [6]:
import zipfile
import os

zip_path = "archive (1).zip"
extract_path = "/content/dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [7]:
os.listdir("/content/dataset")

['fertilizer_recommendation.csv']

In [8]:
import pandas as pd

df = pd.read_csv("/content/dataset/fertilizer_recommendation.csv")
df.head()

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Nitrogen_Level,Phosphorus_Level,Potassium_Level,Temperature,Humidity,Rainfall,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Previous_Crop,Region,Fertilizer_Used_Last_Season,Yield_Last_Season,Recommended_Fertilizer
0,Clay,6.07,34.98,0.32,1.87,61,44,84,19.84,83.31,1693.22,Cotton,Harvest,Kharif,Canal,Wheat,South,297.15,1.19,MOP
1,Silt,6.39,47.34,0.28,0.21,59,56,18,24.40,46.27,1030.21,Maize,Vegetative,Kharif,Sprinkler,Potato,Central,77.17,4.40,Urea
2,Sandy,7.92,38.13,0.99,1.88,43,21,119,24.82,71.86,1166.16,Cotton,Flowering,Kharif,Rainfed,Tomato,South,128.93,7.21,Urea
3,Clay,5.86,14.17,1.46,0.36,88,46,34,27.87,53.23,2881.83,Wheat,Flowering,Zaid,Sprinkler,Potato,West,233.96,1.85,MOP
4,Clay,7.98,19.28,0.85,2.16,104,53,98,24.17,51.87,714.84,Potato,Sowing,Kharif,Rainfed,Maize,East,214.39,7.36,Zinc Sulphate


In [9]:
!pip install lightgbm optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 7.9 MB/s eta 0:00:00


In [10]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

import lightgbm as lgb
import optuna

In [11]:
df = pd.read_csv("/content/dataset/fertilizer_recommendation.csv")
df.head()

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Nitrogen_Level,Phosphorus_Level,Potassium_Level,Temperature,Humidity,Rainfall,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Previous_Crop,Region,Fertilizer_Used_Last_Season,Yield_Last_Season,Recommended_Fertilizer
0,Clay,6.07,34.98,0.32,1.87,61,44,84,19.84,83.31,1693.22,Cotton,Harvest,Kharif,Canal,Wheat,South,297.15,1.19,MOP
1,Silt,6.39,47.34,0.28,0.21,59,56,18,24.40,46.27,1030.21,Maize,Vegetative,Kharif,Sprinkler,Potato,Central,77.17,4.40,Urea
2,Sandy,7.92,38.13,0.99,1.88,43,21,119,24.82,71.86,1166.16,Cotton,Flowering,Kharif,Rainfed,Tomato,South,128.93,7.21,Urea
3,Clay,5.86,14.17,1.46,0.36,88,46,34,27.87,53.23,2881.83,Wheat,Flowering,Zaid,Sprinkler,Potato,West,233.96,1.85,MOP
4,Clay,7.98,19.28,0.85,2.16,104,53,98,24.17,51.87,714.84,Potato,Sowing,Kharif,Rainfed,Maize,East,214.39,7.36,Zinc Sulphate


In [12]:
print(df['Crop_Type'].unique())

['Cotton' 'Maize' 'Wheat' 'Potato' 'Rice' 'Sugarcane' 'Tomato']


In [13]:
TARGET = "Recommended_Fertilizer"

In [14]:
DROP_COLS = [
    "Yield_Last_Season",
    "Fertilizer_Used_Last_Season"
]

df = df.drop(columns=DROP_COLS, errors="ignore")

In [15]:
categorical_features = [
    "Soil_Type",
    "Crop_Type",
    "Crop_Growth_Stage",
    "Season",
    "Irrigation_Type",
    "Previous_Crop",
    "Region"
]

numerical_features = [
    "Soil_pH",
    "Soil_Moisture",
    "Organic_Carbon",
    "Electrical_Conductivity",
    "Nitrogen_Level",
    "Phosphorus_Level",
    "Potassium_Level",
    "Temperature",
    "Humidity",
    "Rainfall"
]

In [16]:
# Numerical → median
for col in numerical_features:
    df[col] = df[col].fillna(df[col].median())

# Categorical → mode
for col in categorical_features:
    df[col] = df[col].fillna(df[col].mode()[0])

In [17]:
label_encoders = {}

for col in categorical_features:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

In [18]:
df["NPK_Ratio"] = (
    df["Nitrogen_Level"] +
    df["Phosphorus_Level"] +
    df["Potassium_Level"]
) / 3

In [19]:
df["Moisture_Rain_Interaction"] = df["Soil_Moisture"] * df["Rainfall"]

In [20]:
df["Climate_Stress"] = df["Temperature"] / (df["Humidity"] + 1)

In [21]:
from sklearn.preprocessing import LabelEncoder

target_encoder = LabelEncoder()
df[TARGET] = target_encoder.fit_transform(df[TARGET])

df[TARGET].head()

,Recommended_Fertilizer
0,2
1,5
2,5
3,2
4,6


In [22]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [23]:
def objective(trial):

    params = {
        "objective": "multiclass",
        "num_class": y.nunique(),
        "metric": "multi_logloss",
        "boosting_type": "gbdt",
        "verbosity": -1,

        # REQUIRED FOR OPTUNA + min_data_in_leaf
        "feature_pre_filter": False,

        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15),
        "num_leaves": trial.suggest_int("num_leaves", 20, 300),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 100),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 5.0),
        "lambda_l2": trial.suggest_float("lambda_l2", 0.0, 5.0)
    }

    #  CREATE DATASET INSIDE OBJECTIVE
    train_data = lgb.Dataset(X_train, label=y_train)
    valid_data = lgb.Dataset(X_test, label=y_test)

    model = lgb.train(
        params,
        train_data,
        valid_sets=[valid_data],
        num_boost_round=300,
        callbacks=[lgb.early_stopping(30, verbose=False)]
    )

    preds = model.predict(X_test)
    preds = np.argmax(preds, axis=1)

    return accuracy_score(y_test, preds)

In [24]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

study.best_value, study.best_params

[I 2026-02-28 17:01:42,940] A new study created in memory with name: no-name-7cba7ad6-f2a7-438d-a9aa-e9dbd938684f
[I 2026-02-28 17:01:45,696] Trial 0 finished with value: 0.8735 and parameters: {'learning_rate': 0.11839611632148617, 'num_leaves': 62, 'max_depth': 12, 'min_data_in_leaf': 40, 'feature_fraction': 0.9084386695667241, 'bagging_fraction': 0.753441868634651, 'bagging_freq': 2, 'lambda_l1': 0.6619404743010698, 'lambda_l2': 1.235577367112476}. Best is trial 0 with value: 0.8735.
[I 2026-02-28 17:01:48,758] Trial 1 finished with value: 0.875 and parameters: {'learning_rate': 0.03865715775129701, 'num_leaves': 299, 'max_depth': 11, 'min_data_in_leaf': 26, 'feature_fraction': 0.7524589957399632, 'bagging_fraction': 0.6587673715078766, 'bagging_freq': 4, 'lambda_l1': 1.2741433613776048, 'lambda_l2': 0.7783599283175402}. Best is trial 1 with value: 0.875.
[I 2026-02-28 17:01:50,174] Trial 2 finished with value: 0.876 and parameters: {'learning_rate': 0.07550890399243483, 'num_leaves

(0.878,
 {'learning_rate': 0.06769733483799212,
  'num_leaves': 260,
  'max_depth': 8,
  'min_data_in_leaf': 64,
  'feature_fraction': 0.7793985652709912,
  'bagging_fraction': 0.7178011321870961,
  'bagging_freq': 6,
  'lambda_l1': 3.4837463119053416,
  'lambda_l2': 0.41341711303092843})

In [25]:
study.best_value
study.best_params

{'learning_rate': 0.06769733483799212,
 'num_leaves': 260,
 'max_depth': 8,
 'min_data_in_leaf': 64,
 'feature_fraction': 0.7793985652709912,
 'bagging_fraction': 0.7178011321870961,
 'bagging_freq': 6,
 'lambda_l1': 3.4837463119053416,
 'lambda_l2': 0.41341711303092843}

In [26]:
best_params = study.best_params
best_params.update({
    "objective": "multiclass",
    "num_class": y.nunique(),
    "metric": "multi_logloss",
    "verbosity": -1
})

final_model = lgb.train(
    best_params,
    lgb.Dataset(X_train, label=y_train),
    num_boost_round=500
)

In [27]:
study.best_value

0.878

In [28]:
from sklearn.metrics import accuracy_score

y_pred = final_model.predict(X_test)
y_pred = np.argmax(y_pred, axis=1)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy Score:", accuracy)

Accuracy Score: 0.8785


In [29]:
import joblib

joblib.dump(final_model, "fertilizer_model.pkl")
joblib.dump(target_encoder, "target_encoder.pkl")
joblib.dump(label_encoders, "feature_encoders.pkl")

['feature_encoders.pkl']

In [30]:
fertilizer_composition = {
    "Urea": {"N": 0.46, "P": 0.0, "K": 0.0},
    "DAP": {"N": 0.18, "P": 0.46, "K": 0.0},
    "MOP": {"N": 0.0, "P": 0.0, "K": 0.60},
    "NPK_10_26_26": {"N": 0.10, "P": 0.26, "K": 0.26}
}

In [31]:
crop_requirements = {
    "Cotton": {"N": 150, "P": 60, "K": 60},
    "Maize": {"N": 150, "P": 70, "K": 60},
    "Wheat": {"N": 120, "P": 60, "K": 40},
    "Potato": {"N": 180, "P": 80, "K": 100},
    "Rice": {"N": 100, "P": 50, "K": 50},
    "Sugarcane": {"N": 250, "P": 100, "K": 120},
    "Tomato": {"N": 140, "P": 60, "K": 60}
}

In [32]:
import numpy as np
import pandas as pd

def predict_fertilizer(new_data_dict):

    # Convert input dictionary to DataFrame
    new_df = pd.DataFrame([new_data_dict])

    # ----------------------------
    # Feature Engineering (Same as Training)
    # ----------------------------

    new_df["NPK_Ratio"] = (
        new_df["Nitrogen_Level"] +
        new_df["Phosphorus_Level"] +
        new_df["Potassium_Level"]
    ) / 3

    new_df["Moisture_Rain_Interaction"] = (
        new_df["Soil_Moisture"] * new_df["Rainfall"]
    )

    new_df["Climate_Stress"] = (
        new_df["Temperature"] / (new_df["Humidity"] + 1)
    )

    # ----------------------------
    # Encode Categorical Features
    # ----------------------------

    for col in categorical_features:
        if new_df[col][0] in label_encoders[col].classes_:
            new_df[col] = label_encoders[col].transform(new_df[col])
        else:
            new_df[col] = 0  # fallback for unseen category

    # ----------------------------
    # Predict Fertilizer Type
    # ----------------------------

    prediction = final_model.predict(new_df)
    prediction = np.argmax(prediction, axis=1)

    fertilizer_name = target_encoder.inverse_transform(prediction)[0]

    # ----------------------------
    # Quantity Calculation (Rule-Based)
    # ----------------------------

    crop_name = new_data_dict["Crop_Type"]
    soil_N = new_data_dict["Nitrogen_Level"]
    soil_P = new_data_dict["Phosphorus_Level"]
    soil_K = new_data_dict["Potassium_Level"]

    # Get crop requirement
    crop_req = crop_requirements.get(crop_name, {"N": 0, "P": 0, "K": 0})

    # Calculate nutrient deficit
    N_deficit = max(0, crop_req["N"] - soil_N)
    P_deficit = max(0, crop_req["P"] - soil_P)
    K_deficit = max(0, crop_req["K"] - soil_K)

    # Get fertilizer composition
    fert_comp = fertilizer_composition.get(fertilizer_name, {"N": 0, "P": 0, "K": 0})

    fertilizer_quantity = 0

    if fert_comp["N"] > 0:
        fertilizer_quantity = N_deficit / fert_comp["N"]
    elif fert_comp["P"] > 0:
        fertilizer_quantity = P_deficit / fert_comp["P"]
    elif fert_comp["K"] > 0:
        fertilizer_quantity = K_deficit / fert_comp["K"]

    fertilizer_quantity = round(fertilizer_quantity, 2)

    # ----------------------------
    # Final Output
    # ----------------------------

    return {
        "Recommended_Fertilizer": fertilizer_name,
        "Recommended_Quantity_kg_per_hectare": fertilizer_quantity
    }

In [33]:
new_farmer_data = {
    "Soil_Type": "hehe",
    "Soil_pH": 6.5,
    "Soil_Moisture": 35,
    "Organic_Carbon": 0.8,
    "Electrical_Conductivity": 0.3,
    "Nitrogen_Level": 50,
    "Phosphorus_Level": 40,
    "Potassium_Level": 45,
    "Temperature": 28,
    "Humidity": 65,
    "Rainfall": 120,
    "Crop_Type": "Wheat",
    "Crop_Growth_Stage": "Vegetative",
    "Season": "Rabi",
    "Irrigation_Type": "Drip",
    "Previous_Crop": "Rice",
    "Region": "North"
}

In [34]:
predict_fertilizer(new_farmer_data)

{'Recommended_Fertilizer': 'Urea',
 'Recommended_Quantity_kg_per_hectare': 152.17}

In [35]:
import joblib
import pandas as pd
import numpy as np

# Load artifacts
model = joblib.load('fertilizer_model.pkl')
target_encoder = joblib.load('target_encoder.pkl')
feature_encoders = joblib.load('feature_encoders.pkl')  # Dict of LabelEncoders

# Critical: Must match training configuration
CATEGORICAL_FEATURES = [
    "Soil_Type", "Crop_Type", "Crop_Growth_Stage",
    "Season", "Irrigation_Type", "Previous_Crop", "Region"
]
def calculate_fertilizer_quantity(input_data, fertilizer_type):
    """
    Simple rule-based fertilizer quantity recommendation (kg/hectare)
    Based on crop type and NPK deficiency
    """

    crop = input_data["Crop_Type"]

    # Ideal NPK requirement (simple agricultural approximation per hectare)
    crop_npk_requirement = {
        "Rice": 120,
        "Wheat": 100,
        "Maize": 110,
        "Cotton": 90,
        "Sugarcane": 150,
        "Potato": 130,
        "Tomato": 115
    }

    if crop not in crop_npk_requirement:
        return 50  # safe fallback

    ideal_npk = crop_npk_requirement[crop]

    # Current average soil nutrient level
    current_npk = (
        input_data["Nitrogen_Level"] +
        input_data["Phosphorus_Level"] +
        input_data["Potassium_Level"]
    ) / 3

    # Deficiency
    deficiency = max(0, ideal_npk - current_npk)

    # Simple scaling factor (avoid overuse)
    recommended_qty = round(deficiency * 0.8, 2)

    return recommended_qty

def predict_fertilizer(input_data: dict) -> dict:
    """
    Returns:
        {
            "fertilizer_type": str,
            "recommended_quantity_kg_per_hectare": float
        }
    """

    required_cols = [
        'Soil_Type', 'Soil_pH', 'Soil_Moisture', 'Organic_Carbon',
        'Electrical_Conductivity', 'Nitrogen_Level', 'Phosphorus_Level',
        'Potassium_Level', 'Temperature', 'Humidity', 'Rainfall',
        'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type',
        'Previous_Crop', 'Region'
    ]

    if not all(col in input_data for col in required_cols):
        missing = [col for col in required_cols if col not in input_data]
        raise ValueError(f"Missing required columns: {missing}")

    df = pd.DataFrame([input_data])

    # Feature Engineering (same as training)
    df["NPK_Ratio"] = (
        df["Nitrogen_Level"] +
        df["Phosphorus_Level"] +
        df["Potassium_Level"]
    ) / 3

    df["Moisture_Rain_Interaction"] = df["Soil_Moisture"] * df["Rainfall"]
    df["Climate_Stress"] = df["Temperature"] / (df["Humidity"] + 1)

    # Encode categoricals safely
    for col in CATEGORICAL_FEATURES:
        if df[col].iloc[0] in feature_encoders[col].classes_:
            df[col] = feature_encoders[col].transform(df[col])
        else:
            df[col] = feature_encoders[col].transform(
                [feature_encoders[col].classes_[0]]
            )[0]

    # Match feature order
    expected_features = model.feature_name()
    df = df[expected_features]

    # Predict fertilizer class
    pred_idx = np.argmax(model.predict(df), axis=1)
    fertilizer_name = target_encoder.inverse_transform(pred_idx)[0]

    # Calculate quantity using rule-based logic
    quantity = calculate_fertilizer_quantity(input_data, fertilizer_name)

    return {
        "fertilizer_type": fertilizer_name,
        "recommended_quantity_kg_per_hectare": quantity
    }

# Example usage
if __name__ == "__main__":
    sample_input = {
        "Soil_Type": "silt", "Soil_pH": 4.67, "Soil_Moisture": 35.58,
        "Organic_Carbon": 1.35, "Electrical_Conductivity": 0.74,
        "Nitrogen_Level": 102, "Phosphorus_Level": 27, "Potassium_Level": 44,
        "Temperature": 30.88, "Humidity": 83.02, "Rainfall": 1129.91,
        "Crop_Type": "Rice", "Crop_Growth_Stage": "Sowing",
        "Season": "Kharif", "Irrigation_Type": "Rainfed",
        "Previous_Crop": "Cotton", "Region": "North"
    }

    result = predict_fertilizer(sample_input)

    print(f"🌱 Recommended Fertilizer: {result['fertilizer_type']}")
    print(f"📦 Quantity (kg/hectare): {result['recommended_quantity_kg_per_hectare']}")

🌱 Recommended Fertilizer: DAP
📦 Quantity (kg/hectare): 49.87


In [36]:
import pandas as pd

importance = final_model.feature_importance(importance_type="gain")

feature_importance_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": importance
}).sort_values(by="Importance", ascending=False)

feature_importance_df

,Feature,Importance
5,Nitrogen_Level,45481.154687
6,Phosphorus_Level,31890.204061
7,Potassium_Level,16924.020905
1,Soil_pH,16383.524608
17,NPK_Ratio,11967.796211
12,Crop_Growth_Stage,3106.097185
4,Electrical_Conductivity,1381.045082
10,Rainfall,1325.806950
3,Organic_Carbon,1321.226183
2,Soil_Moisture,1255.378816


You can list all the files in the current directory (`/content/`) using the `!ls` command:

In [37]:
import os
print(os.listdir('/content'))

['.config', 'feature_encoders.pkl', 'archive.zip', 'target_encoder.pkl', 'fertilizer_model.pkl', 'dataset', 'archive (1).zip', 'sample_data']


To download the `fertilizer_model.pkl` file, you can use the `files.download` function from `google.colab`:

In [38]:
from google.colab import files

files.download('fertilizer_model.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

You can repeat the `files.download()` command for `target_encoder.pkl` and `feature_encoders.pkl` as well:

In [39]:
files.download('target_encoder.pkl')
files.download('feature_encoders.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Combine features + target
df_corr = df.copy()

corr_matrix = df_corr.corr()

In [ ]:
plt.figure(figsize=(14,10))
sns.heatmap(
    corr_matrix,
    cmap="coolwarm",
    annot=False,
    linewidths=0.5
)
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
import shap

explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X_train)

shap.summary_plot(shap_values, X_train)

In [ ]:
shap.summary_plot(shap_values, X_train, plot_type="bar")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

for col in numerical_features:
    plt.figure(figsize=(8,5))
    sns.boxplot(
        x=df[TARGET],
        y=df[col]
    )
    plt.title(f"{col} vs {TARGET}")
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
for col in categorical_features:
    plt.figure(figsize=(8,5))
    sns.countplot(
        data=df,
        x=col,
        hue=TARGET
    )
    plt.title(f"{col} vs {TARGET}")
    plt.xticks(rotation=45)
    plt.show()